In [72]:
import pandas as pd
from collections import Counter
import math
import ast
import datasets
import os

In [73]:
# Path to the .csv generated by the script extract.py
csv_file = "/projects/iris/rferreir/GeoBenchmark/data/GeoQuestions1089/dataset.csv"

# Path to the output files
output_file = "/projects/iris/rferreir/GeoBenchmark/data/GeoQuestions1089/dataset2.csv"

# Path to the directory where to save the HF datasets
output_path = "/projects/iris/rferreir/hub_datasets"

In [51]:
data = pd.read_csv(csv_file)
print(len(data))
data["question_id"] = range(1, len(data) + 1)
data = data[data["cleaned_answer"] != "[]"]
print(len(data))
data.head()

1017
954


,question,cleaned_answer,category,original_answer,answer_type,question_id
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,['numeric'],1
1,Where is the Dorset county located?,"[(50.80540690860262, -2.3225184770895906)]",A,"wkt\r\n""POLYGON ((-2.65268492035382 50.6698994...",['coord'],2
2,Where is Lough Ramor located?,"[(53.81842561485053, -7.082299497623002)]",A,"wkt\r\n""POLYGON ((-7.1289069000000005 53.82470...",['coord'],3
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,['numeric'],4
4,Where is Oxfordshire located?,"[(51.76576557729234, -1.3223245467838023)]",A,"WKT\r\n""POLYGON ((-1.6028258490004 51.51829434...",['coord'],5


## Convert square degrees to square km 

In [52]:
mots = ['area', 'size']

def check_for_degrees(row):
    a = ast.literal_eval(row['cleaned_answer'])
    if 'area' in row['question'] or 'size' in row['question'] or 'How large' in row['question']:
        try:
            v = float(a[0])
            if v % 1 != 0:
                row['answer_type'] = 'sq degrees'
        except:
            print(row['question'])
    return row

data.apply(check_for_degrees, axis=1)

Which rural areas border with Castleblayney?
Which rural areas border with Clonakilty?
Which is the largest park by area in the municipality of Athens?
Identify any nature reserves that are situated south of a forested area.
Which rural areas intersect with lakes?
Which lakes are linked to a woodland area?
Which forests intersect with rural areas in the Republic of Ireland?
Which rural areas located south of County Longford have forests?
Which rural areas are located on the northern side of the city of Cork and have forests?
Can you identify the woodland areas in the districts situated to the south of Staffordshire?
What is the name and the area of the parks that are in Wards of Northern Ireland that are east of Dublin?
Which local government districts in Northern Ireland, north of Belfast, have an area of more than 1000000 square meters?
What is the area of each forest in the Scarawalsh Barony?
Which towns in County Longford with area bigger than 0.01 are west of Dublin?
Which county 

,question,cleaned_answer,category,original_answer,answer_type,question_id
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,['numeric'],1
1,Where is the Dorset county located?,"[(50.80540690860262, -2.3225184770895906)]",A,"wkt\r\n""POLYGON ((-2.65268492035382 50.6698994...",['coord'],2
2,Where is Lough Ramor located?,"[(53.81842561485053, -7.082299497623002)]",A,"wkt\r\n""POLYGON ((-7.1289069000000005 53.82470...",['coord'],3
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,['numeric'],4
4,Where is Oxfordshire located?,"[(51.76576557729234, -1.3223245467838023)]",A,"WKT\r\n""POLYGON ((-1.6028258490004 51.51829434...",['coord'],5
...,...,...,...,...,...,...
1012,In which state is the tallest building in New ...,['New York World Building'],H,building\r\nhttp://yago-knowledge.org/resource...,['str'],1013
1013,Which is the tallest building in Chicago?,['Masonic Temple (Chicago'],H,"building\r\n""http://yago-knowledge.org/resourc...",['str'],1014
1014,What is the total population of the 5 biggest ...,[104496392985.0],I,totalpop\r\n104496392985\r\n,['numeric'],1015
1015,What is the population total of the 5 largest ...,[9685744.0],I,populationTotal\r\n9685744\r\n,['numeric'],1016


In [53]:
print(len(data[data['answer_type'] == 'sq degrees']))
#print(data[data['answer_type']== 'sq degrees']['question'])

0


In [54]:
latitude = 48.8566
km2_par_deg2 = 111.32 * 111.32 * math.cos(math.radians(latitude))
def change_degrees_in_km(row):
    if row['answer_type'] == 'sq degrees':
        values = ast.literal_eval(row['cleaned_answer'])
        n_values = []
        for v in values:
            try:
                fv = float(v)
                n_values.append(km2_par_deg2 * fv)
            except:
                continue
        row['cleaned_answer'] = n_values
    return row

data.apply(change_degrees_in_km, axis=1)

,question,cleaned_answer,category,original_answer,answer_type,question_id
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,['numeric'],1
1,Where is the Dorset county located?,"[(50.80540690860262, -2.3225184770895906)]",A,"wkt\r\n""POLYGON ((-2.65268492035382 50.6698994...",['coord'],2
2,Where is Lough Ramor located?,"[(53.81842561485053, -7.082299497623002)]",A,"wkt\r\n""POLYGON ((-7.1289069000000005 53.82470...",['coord'],3
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,['numeric'],4
4,Where is Oxfordshire located?,"[(51.76576557729234, -1.3223245467838023)]",A,"WKT\r\n""POLYGON ((-1.6028258490004 51.51829434...",['coord'],5
...,...,...,...,...,...,...
1012,In which state is the tallest building in New ...,['New York World Building'],H,building\r\nhttp://yago-knowledge.org/resource...,['str'],1013
1013,Which is the tallest building in Chicago?,['Masonic Temple (Chicago'],H,"building\r\n""http://yago-knowledge.org/resourc...",['str'],1014
1014,What is the total population of the 5 biggest ...,[104496392985.0],I,totalpop\r\n104496392985\r\n,['numeric'],1015
1015,What is the population total of the 5 largest ...,[9685744.0],I,populationTotal\r\n9685744\r\n,['numeric'],1016


In [55]:
print(data[data['answer_type'] == 'sq degrees'][['question', 'cleaned_answer']])
data.loc[data['answer_type'] == "sq degrees", 'answer_type'] = 'sq km'

Empty DataFrame
Columns: [question, cleaned_answer]
Index: []


## Translating place names from Greek to English

In [56]:
data.loc[data['original_answer'].str.contains('gagentity', case=False, na=False), 'trad'] = 'True'
print(len(data[data['trad'] == 'True']))


7


In [57]:
to_trad = data[data['trad'] == 'True']
def print_row(row):
    print(row['question'])
    print(row['cleaned_answer'])
    print("\n")

to_trad.apply(print_row, axis=1)

Which regional units border with Thessaly?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΙΕΡΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΓΡΕΒΕΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΟΖΑΝΗΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΑΙΤΩΛΟΑΚΑΡΝΑΝΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΦΘΙΩΤΙΔΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΕΥΡΥΤΑΝΙΑΣ']


Which regional units border the regional unit of Thessaloniki?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΣΕΡΡΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΕΛΛΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΙΛΚΙΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΗΜΑΘΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΧΑΛΚΙΔΙΚΗΣ']


Which are Thessaly's neighboring Regional Units?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΙΕΡΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΓΡΕΒΕΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΚΟΖΑΝΗΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΑΙΤΩΛΟΑΚΑΡΝΑΝΙΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΦΘΙΩΤΙΔΑΣ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΕΥΡΥΤΑΝΙΑΣ']


Which regional unit in the region of Epirus has a lake?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΙΩΑΝΝΙΝΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΠΡΕΒΕΖΑΣ']


Which regional unit in the region of Thessaly has a beach?
['ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΣΠΟΡΑΔΩΝ', 'ΠΕΡΙΦΕΡΕΙΑΚΗ ΕΝΟΤΗΤΑ ΜΑΓΝΗΣΙΑΣ']


Which r

286    None
326    None
391    None
475    None
547    None
576    None
604    None
dtype: object

In [58]:
a = "['REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF GREVENA', 'REGIONAL UNIT OF KOZANI', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FTHIOTIDA', 'REGIONAL UNIT OF EVRYTANIA']"
a = a.replace("REGIONAL UNIT OF ", '')
data.loc[data['question'] == 'Which regional units border with Thessaly?', 'cleaned_answer'] = a

In [59]:
a = "['REGIONAL UNIT OF SERRA', 'REGIONAL UNIT OF PELLA', 'REGIONAL UNIT OF KILKIS', 'REGIONAL UNIT OF IMATHIA', 'REGIONAL UNIT OF HALKIDIKI']"
a = a.replace("REGIONAL UNIT OF ", '')
data.loc[data['question'] == 'Which regional units border the regional unit of Thessaloniki?', 'cleaned_answer'] = a

In [60]:
a = "['REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF GREVENA', 'REGIONAL UNIT OF KOZANI', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FTHIOTIDA', 'REGIONAL UNIT OF EVRYTANIA']"
a = a.replace("REGIONAL UNIT OF ", '')
data.loc[data['question'] == "Which are Thessaly's neighboring Regional Units?", 'cleaned_answer'] = a

In [61]:
a = "['REGIONAL UNIT OF IOANNINA', 'REGIONAL UNIT OF PREVEZA']"
a = a.replace("REGIONAL UNIT OF ", '')
data.loc[data['question'] == "Which regional unit in the region of Epirus has a lake?", 'cleaned_answer'] = a

In [62]:
a = "['SPORADES REGIONAL UNIT', 'MAGNESIA REGIONAL UNIT']"
a = a.replace(" REGIONAL UNIT", '')
data.loc[data['question'] == "Which regional unit in the region of Thessaly has a beach?", 'cleaned_answer'] = a

In [63]:
a = "['REGIONAL UNIT OF SERRA', 'REGIONAL UNIT OF PIERIA', 'REGIONAL UNIT OF KILKIS', 'REGIONAL UNIT OF KASTORIA', 'REGIONAL UNIT OF FLORINA', 'REGIONAL UNIT OF PREVEZA', 'REGIONAL UNIT OF IOANNINA', 'REGIONAL UNIT OF TRIKALA', 'REGIONAL UNIT OF ETOLOAKARNANIA', 'REGIONAL UNIT OF FOKIDA', 'REGIONAL UNIT OF VIOTIA']"
a = a.replace("REGIONAL UNIT OF ", '')
data.loc[data['question'] == "Which regional units of Greece share a border with three regional units?", 'cleaned_answer'] = a

In [64]:
a = "['SPORADES REGIONAL UNIT', 'ITHAKI REGIONAL UNIT', 'IKARIAS REGIONAL UNIT', 'Perifereiakí Enótita Mykónou', 'Perifereiakí Enótita Tínou', 'Perifereiakí Enótita Karpáthou', 'Perifereiakí Enótita Kéas - Kýthnou', 'Perifereiakí Enótita Mílou']"
a = a.replace(" REGIONAL UNIT", '')
data.loc[data['question'] == "Which Greek regional units have less than 10000 inhabitants?", 'cleaned_answer'] = a

In [65]:
to_trad = data[data['trad'] == 'True']
def print_row(row):
    print(row['question'])
    print(row['cleaned_answer'])
    print("\n")

to_trad.apply(print_row, axis=1)

Which regional units border with Thessaly?
['PIERIA', 'GREVENA', 'KOZANI', 'ETOLOAKARNANIA', 'FTHIOTIDA', 'EVRYTANIA']


Which regional units border the regional unit of Thessaloniki?
['SERRA', 'PELLA', 'KILKIS', 'IMATHIA', 'HALKIDIKI']


Which are Thessaly's neighboring Regional Units?
['PIERIA', 'GREVENA', 'KOZANI', 'ETOLOAKARNANIA', 'FTHIOTIDA', 'EVRYTANIA']


Which regional unit in the region of Epirus has a lake?
['IOANNINA', 'PREVEZA']


Which regional unit in the region of Thessaly has a beach?
['SPORADES', 'MAGNESIA']


Which regional units of Greece share a border with three regional units?
['SERRA', 'PIERIA', 'KILKIS', 'KASTORIA', 'FLORINA', 'PREVEZA', 'IOANNINA', 'TRIKALA', 'ETOLOAKARNANIA', 'FOKIDA', 'VIOTIA']


Which Greek regional units have less than 10000 inhabitants?
['SPORADES', 'ITHAKI', 'IKARIAS', 'Perifereiakí Enótita Mykónou', 'Perifereiakí Enótita Tínou', 'Perifereiakí Enótita Karpáthou', 'Perifereiakí Enótita Kéas - Kýthnou', 'Perifereiakí Enótita Mílou']




286    None
326    None
391    None
475    None
547    None
576    None
604    None
dtype: object

In [66]:
data['cleaned_answer'] = data['cleaned_answer'].str.replace('%27', "'", regex=False)

In [67]:
data.to_csv(output_file, index=False)

## Converting to HF dataset

In [68]:
data.head()

,question,cleaned_answer,category,original_answer,answer_type,question_id,trad
0,What is the population of Central Greece?,[570922.0],A,population\r\n570922\r\n,['numeric'],1,NaN
1,Where is the Dorset county located?,"[(50.80540690860262, -2.3225184770895906)]",A,"wkt\r\n""POLYGON ((-2.65268492035382 50.6698994...",['coord'],2,NaN
2,Where is Lough Ramor located?,"[(53.81842561485053, -7.082299497623002)]",A,"wkt\r\n""POLYGON ((-7.1289069000000005 53.82470...",['coord'],3,NaN
3,What is the population of the Municipality of ...,[38116.0],A,population\r\n38116\r\n,['numeric'],4,NaN
4,Where is Oxfordshire located?,"[(51.76576557729234, -1.3223245467838023)]",A,"WKT\r\n""POLYGON ((-1.6028258490004 51.51829434...",['coord'],5,NaN


In [70]:
to_keep = ['question_id', 'question', 'answer', 'answer_type']
def get_cat(row):
    t = row['answer_type'].replace('[', '').replace(']', '').replace("'", "").split(',')
    if t[0] == 'sq km':
        t[0] = 'numeric'
    row['cat'] = t[0]
    return row

data = data.apply(get_cat, axis=1)

print(data['cat'].value_counts())

yesno = data[data["cat"] == "bool"]

coord = data[data["cat"] == "coord"]

place = data[data["cat"] == "str"]

regression = data[data["cat"] == "numeric"]

cat
str        455
numeric    234
bool       181
coord       84
Name: count, dtype: int64


In [74]:
map_cat_to_data = {
    'YN': yesno,
    'coord': coord,
    'place': place,
    'regression': regression
}

d1 = {}

for name, data_csv in map_cat_to_data.items():
    test = datasets.Dataset.from_pandas(data_csv)
    d1[name] = datasets.DatasetDict({
            "test": test
        })
    d1[name].save_to_disk(os.path.join(output_path, f"GeoQuestions1089_{name}"))

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 99.01ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 10.6kB / 10.6kB, 17.7kB/s  
Processing Files (1 / 1): 100%|██████████| 10.6kB / 10.6kB, 13.2kB/s  
New Data Upload: 100%|██████████| 10.6kB / 10.6kB, 13.2kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  3.29ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   2%|▏         |  746kB / 32.5MB, 1.86MB/s  
Processing Files (0 / 1):   5%|▍         | 1.49MB / 32.5MB, 2.48MB/s  
Processing Files (0 / 1):  16%|█▌        | 5.22MB / 32.5MB, 6.52MB/s  
Processing Files (0 / 1):  28%|██▊       | 8.95MB / 32.5MB, 8.94MB/s  
Processing Files (0 / 1):  37%|███▋      | 11.9MB / 32.5MB, 9.94MB/s  
Processing Files (0 / 1):  44%|████▎     | 14.2MB / 32.5MB, 10.1MB/s  
Processing Files (0 / 1):  57%|█████▋    | 18.6MB / 32.5MB, 11.6MB/s  
P

## Sanity Check

In [75]:
import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

for name in d1.keys():
    d2 = datasets.load_dataset("rfr2003/Geo_Benchmark", f"GeoQuestions1089_{name}")
    assert content_hash(d1[name]['test']) == content_hash(d2['test'])

Generating test split: 100%|██████████| 234/234 [00:00<00:00, 37272.79 examples/s]
